# Phase 3 — joint SASV score fusion & integrated voice security pipeline

## Where we are

| Phase | Model | Headline | Deployed threshold |
|---|---|---|---|
| **Phase 1** | ECAPA-TDNN (`spkrec-ecapa-voxceleb`) | 192-dim speaker embedding, VoxCeleb1 cosine trials | fixed cosine threshold |
| **Phase 2e** | SSL-AASIST + layer weighting, telephony graining | ITW EER **7.89%**, ITW FRR 2.9% at threshold | calibrated LLR |
| **Phase 2f** | 2e + top-4 XLS-R unfreeze + real codecs | VAL actDCF ≈ 0.13, β = 1.90 | calibrated LLR, τ<sub>Bayes</sub> = −log β = **−0.642** |
| **Phase 3** | ECAPA-TDNN + SSL-AASIST joint SASV | `actual a-DCF` & `t-EER` | calibrated joint LLR threshold |

## What this notebook does

Following the ASVspoof 5 evaluation protocol and the Phase 3 roadmap (v2.0):

1. **Retires oracle EER as the production target.** Decisions are made on calibrated
   Log-Likelihood Ratios (LLR = a·score + b, prior-weighted logistic regression — the same
   `antispoof_metrics.fit_calibration` used by Phase 2f) at a fixed Bayes threshold,
   not a threshold chosen after seeing eval labels.
2. **Dual-level metric framework.**
   - **Level 1 — standalone countermeasure (Track 1):** `minDCF`, `actDCF` (β = 1.90),
     `Cllr`, EER — computed by the same unit-tested `antispoof_metrics.py` as Phase 2f.
   - **Level 2 — integrated SASV (Track 2):** `min a-DCF` and `actual a-DCF` over
     target / non-target / spoof trial triplets (α = 1.58, γ = 0.84), plus tandem EER.
3. **Score fusion benchmark** — three joint strategies from the roadmap:
   - **Linear LLR fusion:** s<sub>sasv</sub> = w₁·LLR_asv + w₂·LLR_cm + b, fitted by
     logistic regression on pooled validation trials.
   - **Cascaded min-score gate:** s<sub>sasv</sub> = min(LLR_asv − τ_asv, LLR_cm − τ_cm);
     failsafe — reject if either gate fails.
   - **Non-linear MLP fusion head:** 2-layer MLP over [LLR_asv, LLR_cm] features.
4. **Consumes the newest anti-spoof gate attached** (Phase 2j blend `phase2j_gate_config.json` +
   `ssl_2j_best.pt` if present, else Phase 2i `phase2i_gate_config.json` +
   `ssl_2i_best.pt`, else Phase 2h `phase2h_gate_config.json` + `ssl_2h_best.pt`,
   else Phase 2g `phase2g_gate_config.json` + `ssl_2g_best.pt`,
   else Phase 2f `phase2f_gate_config.json` + `ssl_2f_best.pt`) and the
   Phase 1 ECAPA checkpoint when they are attached as Kaggle inputs; a
   `sasv_trials.csv` (columns `split,enroll,test,type`) can drive **real** scoring —
   either attached as a dataset, or **auto-generated** from attached audio
   (In-the-Wild, VoxCeleb1, ASVspoof 2019 LA) by cell 4b. With neither, the harness
   runs on simulated score triplets matched to Phase 1 / Phase 2f distributions.

In [1]:
# ============================================================
# 0. Setup
# ============================================================
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'   # must precede torch/CUDA init

import sys, glob, json, csv, time, random, hashlib, subprocess, copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import transformers

for pkg in ['pyarrow', 'speechbrain', 'scikit-learn']:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

from sklearn.linear_model import LogisticRegression

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
WORK = '/kaggle/working'
os.makedirs(WORK, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__, '| transformers', transformers.__version__, '| device', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

try:
    from kaggle_secrets import UserSecretsClient
    KAGGLE_API_TOKEN = UserSecretsClient().get_secret('KAGGLE_API_TOKEN')
except Exception:
    KAGGLE_API_TOKEN = os.environ.get('KAGGLE_API_TOKEN')
KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME', 'avinnn')

if KAGGLE_API_TOKEN:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle>=1.8'], check=False)
    os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN
    tok = os.path.expanduser('~/.kaggle/access_token')
    os.makedirs(os.path.dirname(tok), exist_ok=True)
    with open(tok, 'w') as f:
        f.write(KAGGLE_API_TOKEN)
    os.chmod(tok, 0o600)
    print('Kaggle API token configured -> Phase 3 artifacts will be pushed automatically.')
else:
    print('No KAGGLE_API_TOKEN secret -> dataset push disabled (use Save Version).')

def push_dir(path, slug, title, message):
    # Create-or-version a private Kaggle dataset from `path`. Never raises.
    if not KAGGLE_API_TOKEN:
        return
    with open(os.path.join(path, 'dataset-metadata.json'), 'w') as f:
        json.dump({'title': title, 'id': f'{KAGGLE_USERNAME}/{slug}',
                   'licenses': [{'name': 'CC0-1.0'}]}, f)
    res = subprocess.run(['kaggle', 'datasets', 'version', '-p', path, '-m', message],
                         capture_output=True, text=True)
    if res.returncode != 0:
        res = subprocess.run(['kaggle', 'datasets', 'create', '-p', path],
                             capture_output=True, text=True)
    print(f'   pushed -> {KAGGLE_USERNAME}/{slug}' if res.returncode == 0 else
          '   dataset push failed: ' + (res.stderr or res.stdout)[-300:])

torch 2.11.0+cu130 | transformers 5.20.2 | device cuda
GPU: Tesla T4
Kaggle API token configured -> Phase 3 artifacts will be pushed automatically.


In [2]:
%%writefile /kaggle/working/ssl_aasist_net.py
"""SSL-AASIST (wav2vec 2.0 XLS-R 300M + AASIST) without fairseq.

Back-end classes are copied from TakHemlata/SSL_Anti-spoofing model.py (MIT,
Tak et al., Odyssey 2022). The fairseq XLS-R front-end is replaced by an
architecturally identical Hugging Face `Wav2Vec2Model`, and the official
checkpoint's fairseq keys are renamed onto it (`load_ssl_aasist_checkpoint`).
This avoids installing fairseq (broken on Python >= 3.11) and does not need
the 1.2 GB `xlsr2_300m.pt`: every weight comes from the anti-spoofing checkpoint.

Input : float32 waveform, 16 kHz mono, shape (batch, 64600), NOT normalised.
Output: (batch, 2) logits; score = logits[:, 1] (higher = more bona fide).
"""
import re
from typing import Union

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import Wav2Vec2Config, Wav2Vec2Model

NB_SAMP = 64600

# XLS-R 300M architecture (facebook/wav2vec2-xls-r-300m config.json); all
# dropout/masking disabled because the anti-spoofing net is used as a scorer.
XLSR_300M_CONFIG = dict(
    hidden_size=1024, num_hidden_layers=24, num_attention_heads=16,
    intermediate_size=4096, hidden_act='gelu',
    conv_dim=(512,) * 7, conv_stride=(5, 2, 2, 2, 2, 2, 2),
    conv_kernel=(10, 3, 3, 3, 3, 2, 2), conv_bias=True,
    feat_extract_norm='layer', feat_extract_activation='gelu',
    do_stable_layer_norm=True, num_conv_pos_embeddings=128,
    num_conv_pos_embedding_groups=16, layer_norm_eps=1e-5,
    apply_spec_augment=False, mask_time_prob=0.0, mask_feature_prob=0.0,
    hidden_dropout=0.0, attention_dropout=0.0, activation_dropout=0.0,
    feat_proj_dropout=0.0, final_dropout=0.0, layerdrop=0.0,
)


class SSLModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = Wav2Vec2Model(Wav2Vec2Config(**XLSR_300M_CONFIG))
        self.out_dim = 1024

    def extract_feat(self, input_data):
        if input_data.ndim == 3:
            input_data = input_data[:, :, 0]
        return self.model(input_data).last_hidden_state   # [batch, frames, 1024]


# ---------------- AASIST back-end (verbatim from SSL_Anti-spoofing) ----------------
class GraphAttentionLayer(nn.Module):
    def __init__(self, in_dim, out_dim, **kwargs):
        super().__init__()
        self.att_proj = nn.Linear(in_dim, out_dim)
        self.att_weight = self._init_new_params(out_dim, 1)
        self.proj_with_att = nn.Linear(in_dim, out_dim)
        self.proj_without_att = nn.Linear(in_dim, out_dim)
        self.bn = nn.BatchNorm1d(out_dim)
        self.input_drop = nn.Dropout(p=0.2)
        self.act = nn.SELU(inplace=True)
        self.temp = kwargs.get('temperature', 1.)

    def forward(self, x):
        x = self.input_drop(x)
        att_map = self._derive_att_map(x)
        x = self._project(x, att_map)
        x = self._apply_BN(x)
        return self.act(x)

    def _pairwise_mul_nodes(self, x):
        nb_nodes = x.size(1)
        x = x.unsqueeze(2).expand(-1, -1, nb_nodes, -1)
        return x * x.transpose(1, 2)

    def _derive_att_map(self, x):
        att_map = self._pairwise_mul_nodes(x)
        att_map = torch.tanh(self.att_proj(att_map))
        att_map = torch.matmul(att_map, self.att_weight)
        att_map = att_map / self.temp
        return F.softmax(att_map, dim=-2)

    def _project(self, x, att_map):
        x1 = self.proj_with_att(torch.matmul(att_map.squeeze(-1), x))
        x2 = self.proj_without_att(x)
        return x1 + x2

    def _apply_BN(self, x):
        org_size = x.size()
        x = x.view(-1, org_size[-1])
        x = self.bn(x)
        return x.view(org_size)

    def _init_new_params(self, *size):
        out = nn.Parameter(torch.FloatTensor(*size))
        nn.init.xavier_normal_(out)
        return out


class HtrgGraphAttentionLayer(nn.Module):
    def __init__(self, in_dim, out_dim, **kwargs):
        super().__init__()
        self.proj_type1 = nn.Linear(in_dim, in_dim)
        self.proj_type2 = nn.Linear(in_dim, in_dim)
        self.att_proj = nn.Linear(in_dim, out_dim)
        self.att_projM = nn.Linear(in_dim, out_dim)
        self.att_weight11 = self._init_new_params(out_dim, 1)
        self.att_weight22 = self._init_new_params(out_dim, 1)
        self.att_weight12 = self._init_new_params(out_dim, 1)
        self.att_weightM = self._init_new_params(out_dim, 1)
        self.proj_with_att = nn.Linear(in_dim, out_dim)
        self.proj_without_att = nn.Linear(in_dim, out_dim)
        self.proj_with_attM = nn.Linear(in_dim, out_dim)
        self.proj_without_attM = nn.Linear(in_dim, out_dim)
        self.bn = nn.BatchNorm1d(out_dim)
        self.input_drop = nn.Dropout(p=0.2)
        self.act = nn.SELU(inplace=True)
        self.temp = kwargs.get('temperature', 1.)

    def forward(self, x1, x2, master=None):
        num_type1 = x1.size(1)
        num_type2 = x2.size(1)
        x1 = self.proj_type1(x1)
        x2 = self.proj_type2(x2)
        x = torch.cat([x1, x2], dim=1)
        if master is None:
            master = torch.mean(x, dim=1, keepdim=True)
        x = self.input_drop(x)
        att_map = self._derive_att_map(x, num_type1, num_type2)
        master = self._update_master(x, master)
        x = self._project(x, att_map)
        x = self._apply_BN(x)
        x = self.act(x)
        x1 = x.narrow(1, 0, num_type1)
        x2 = x.narrow(1, num_type1, num_type2)
        return x1, x2, master

    def _update_master(self, x, master):
        att_map = self._derive_att_map_master(x, master)
        return self._project_master(x, master, att_map)

    def _pairwise_mul_nodes(self, x):
        nb_nodes = x.size(1)
        x = x.unsqueeze(2).expand(-1, -1, nb_nodes, -1)
        return x * x.transpose(1, 2)

    def _derive_att_map_master(self, x, master):
        att_map = x * master
        att_map = torch.tanh(self.att_projM(att_map))
        att_map = torch.matmul(att_map, self.att_weightM)
        att_map = att_map / self.temp
        return F.softmax(att_map, dim=-2)

    def _derive_att_map(self, x, num_type1, num_type2):
        att_map = self._pairwise_mul_nodes(x)
        att_map = torch.tanh(self.att_proj(att_map))
        att_board = torch.zeros_like(att_map[:, :, :, 0]).unsqueeze(-1)
        att_board[:, :num_type1, :num_type1, :] = torch.matmul(
            att_map[:, :num_type1, :num_type1, :], self.att_weight11)
        att_board[:, num_type1:, num_type1:, :] = torch.matmul(
            att_map[:, num_type1:, num_type1:, :], self.att_weight22)
        att_board[:, :num_type1, num_type1:, :] = torch.matmul(
            att_map[:, :num_type1, num_type1:, :], self.att_weight12)
        att_board[:, num_type1:, :num_type1, :] = torch.matmul(
            att_map[:, num_type1:, :num_type1, :], self.att_weight12)
        att_map = att_board / self.temp
        return F.softmax(att_map, dim=-2)

    def _project(self, x, att_map):
        x1 = self.proj_with_att(torch.matmul(att_map.squeeze(-1), x))
        x2 = self.proj_without_att(x)
        return x1 + x2

    def _project_master(self, x, master, att_map):
        x1 = self.proj_with_attM(torch.matmul(att_map.squeeze(-1).unsqueeze(1), x))
        x2 = self.proj_without_attM(master)
        return x1 + x2

    def _apply_BN(self, x):
        org_size = x.size()
        x = x.view(-1, org_size[-1])
        x = self.bn(x)
        return x.view(org_size)

    def _init_new_params(self, *size):
        out = nn.Parameter(torch.FloatTensor(*size))
        nn.init.xavier_normal_(out)
        return out


class GraphPool(nn.Module):
    def __init__(self, k: float, in_dim: int, p: Union[float, int]):
        super().__init__()
        self.k = k
        self.sigmoid = nn.Sigmoid()
        self.proj = nn.Linear(in_dim, 1)
        self.drop = nn.Dropout(p=p) if p > 0 else nn.Identity()
        self.in_dim = in_dim

    def forward(self, h):
        Z = self.drop(h)
        scores = self.sigmoid(self.proj(Z))
        return self.top_k_graph(scores, h, self.k)

    def top_k_graph(self, scores, h, k):
        _, n_nodes, n_feat = h.size()
        n_nodes = max(int(n_nodes * k), 1)
        _, idx = torch.topk(scores, n_nodes, dim=1)
        idx = idx.expand(-1, -1, n_feat)
        h = h * scores
        return torch.gather(h, 1, idx)


class Residual_block(nn.Module):
    def __init__(self, nb_filts, first=False):
        super().__init__()
        self.first = first
        if not self.first:
            self.bn1 = nn.BatchNorm2d(num_features=nb_filts[0])
        self.conv1 = nn.Conv2d(nb_filts[0], nb_filts[1], kernel_size=(2, 3),
                               padding=(1, 1), stride=1)
        self.selu = nn.SELU(inplace=True)
        self.bn2 = nn.BatchNorm2d(num_features=nb_filts[1])
        self.conv2 = nn.Conv2d(nb_filts[1], nb_filts[1], kernel_size=(2, 3),
                               padding=(0, 1), stride=1)
        if nb_filts[0] != nb_filts[1]:
            self.downsample = True
            self.conv_downsample = nn.Conv2d(nb_filts[0], nb_filts[1], padding=(0, 1),
                                             kernel_size=(1, 3), stride=1)
        else:
            self.downsample = False

    def forward(self, x):
        identity = x
        # NB: upstream computes bn1/selu but then feeds `x` to conv1 — kept
        # verbatim so the pretrained weights reproduce the published scores.
        out = self.conv1(x)
        out = self.bn2(out)
        out = self.selu(out)
        out = self.conv2(out)
        if self.downsample:
            identity = self.conv_downsample(identity)
        out += identity
        return out


class SSLAASIST(nn.Module):
    def __init__(self):
        super().__init__()
        filts = [128, [1, 32], [32, 32], [32, 64], [64, 64]]
        gat_dims = [64, 32]
        pool_ratios = [0.5, 0.5, 0.5, 0.5]
        temperatures = [2.0, 2.0, 100.0, 100.0]

        self.ssl_model = SSLModel()
        self.LL = nn.Linear(self.ssl_model.out_dim, 128)
        self.first_bn = nn.BatchNorm2d(num_features=1)
        self.first_bn1 = nn.BatchNorm2d(num_features=64)
        self.drop = nn.Dropout(0.5, inplace=True)
        self.drop_way = nn.Dropout(0.2, inplace=True)
        self.selu = nn.SELU(inplace=True)
        self.encoder = nn.Sequential(
            nn.Sequential(Residual_block(nb_filts=filts[1], first=True)),
            nn.Sequential(Residual_block(nb_filts=filts[2])),
            nn.Sequential(Residual_block(nb_filts=filts[3])),
            nn.Sequential(Residual_block(nb_filts=filts[4])),
            nn.Sequential(Residual_block(nb_filts=filts[4])),
            nn.Sequential(Residual_block(nb_filts=filts[4])))
        self.attention = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=(1, 1)),
            nn.SELU(inplace=True),
            nn.BatchNorm2d(128),
            nn.Conv2d(128, 64, kernel_size=(1, 1)))
        self.pos_S = nn.Parameter(torch.randn(1, 42, filts[-1][-1]))
        self.master1 = nn.Parameter(torch.randn(1, 1, gat_dims[0]))
        self.master2 = nn.Parameter(torch.randn(1, 1, gat_dims[0]))
        self.GAT_layer_S = GraphAttentionLayer(filts[-1][-1], gat_dims[0],
                                               temperature=temperatures[0])
        self.GAT_layer_T = GraphAttentionLayer(filts[-1][-1], gat_dims[0],
                                               temperature=temperatures[1])
        self.HtrgGAT_layer_ST11 = HtrgGraphAttentionLayer(
            gat_dims[0], gat_dims[1], temperature=temperatures[2])
        self.HtrgGAT_layer_ST12 = HtrgGraphAttentionLayer(
            gat_dims[1], gat_dims[1], temperature=temperatures[2])
        self.HtrgGAT_layer_ST21 = HtrgGraphAttentionLayer(
            gat_dims[0], gat_dims[1], temperature=temperatures[2])
        self.HtrgGAT_layer_ST22 = HtrgGraphAttentionLayer(
            gat_dims[1], gat_dims[1], temperature=temperatures[2])
        self.pool_S = GraphPool(pool_ratios[0], gat_dims[0], 0.3)
        self.pool_T = GraphPool(pool_ratios[1], gat_dims[0], 0.3)
        self.pool_hS1 = GraphPool(pool_ratios[2], gat_dims[1], 0.3)
        self.pool_hT1 = GraphPool(pool_ratios[2], gat_dims[1], 0.3)
        self.pool_hS2 = GraphPool(pool_ratios[2], gat_dims[1], 0.3)
        self.pool_hT2 = GraphPool(pool_ratios[2], gat_dims[1], 0.3)
        self.out_layer = nn.Linear(5 * gat_dims[1], 2)

    def forward(self, x):
        x_ssl_feat = self.ssl_model.extract_feat(x.squeeze(-1))
        x = self.LL(x_ssl_feat)
        x = x.transpose(1, 2).unsqueeze(dim=1)
        x = F.max_pool2d(x, (3, 3))
        x = self.first_bn(x)
        x = self.selu(x)
        x = self.encoder(x)
        x = self.first_bn1(x)
        x = self.selu(x)
        w = self.attention(x)

        w1 = F.softmax(w, dim=-1)
        m = torch.sum(x * w1, dim=-1)
        e_S = m.transpose(1, 2) + self.pos_S
        gat_S = self.GAT_layer_S(e_S)
        out_S = self.pool_S(gat_S)

        w2 = F.softmax(w, dim=-2)
        m1 = torch.sum(x * w2, dim=-2)
        e_T = m1.transpose(1, 2)
        gat_T = self.GAT_layer_T(e_T)
        out_T = self.pool_T(gat_T)

        out_T1, out_S1, master1 = self.HtrgGAT_layer_ST11(out_T, out_S, master=self.master1)
        out_S1 = self.pool_hS1(out_S1)
        out_T1 = self.pool_hT1(out_T1)
        out_T_aug, out_S_aug, master_aug = self.HtrgGAT_layer_ST12(out_T1, out_S1, master=master1)
        out_T1 = out_T1 + out_T_aug
        out_S1 = out_S1 + out_S_aug
        master1 = master1 + master_aug

        out_T2, out_S2, master2 = self.HtrgGAT_layer_ST21(out_T, out_S, master=self.master2)
        out_S2 = self.pool_hS2(out_S2)
        out_T2 = self.pool_hT2(out_T2)
        out_T_aug, out_S_aug, master_aug = self.HtrgGAT_layer_ST22(out_T2, out_S2, master=master2)
        out_T2 = out_T2 + out_T_aug
        out_S2 = out_S2 + out_S_aug
        master2 = master2 + master_aug

        out_T1 = self.drop_way(out_T1)
        out_T2 = self.drop_way(out_T2)
        out_S1 = self.drop_way(out_S1)
        out_S2 = self.drop_way(out_S2)
        master1 = self.drop_way(master1)
        master2 = self.drop_way(master2)

        out_T = torch.max(out_T1, out_T2)
        out_S = torch.max(out_S1, out_S2)
        master = torch.max(master1, master2)

        T_max, _ = torch.max(torch.abs(out_T), dim=1)
        T_avg = torch.mean(out_T, dim=1)
        S_max, _ = torch.max(torch.abs(out_S), dim=1)
        S_avg = torch.mean(out_S, dim=1)
        last_hidden = torch.cat([T_max, T_avg, S_max, S_avg, master.squeeze(1)], dim=1)
        last_hidden = self.drop(last_hidden)
        return self.out_layer(last_hidden)


# ---------------- fairseq -> Hugging Face key conversion ----------------
# pre-training-only heads; mask_emb is unused at inference (HF only creates
# masked_spec_embed when masking is enabled, which we disable)
_FAIRSEQ_DROP = ('quantizer.', 'project_q.', 'final_proj.', 'label_embs', 'mask_emb')
_FAIRSEQ_RULES = [
    (r'^feature_extractor\.conv_layers\.(\d+)\.0\.', r'feature_extractor.conv_layers.\1.conv.'),
    (r'^feature_extractor\.conv_layers\.(\d+)\.2\.1\.', r'feature_extractor.conv_layers.\1.layer_norm.'),
    (r'^post_extract_proj\.', 'feature_projection.projection.'),
    (r'^layer_norm\.', 'feature_projection.layer_norm.'),
    (r'^mask_emb$', 'masked_spec_embed'),
    (r'^encoder\.pos_conv\.0\.', 'encoder.pos_conv_embed.conv.'),
    (r'^encoder\.layers\.(\d+)\.self_attn\.', r'encoder.layers.\1.attention.'),
    (r'^encoder\.layers\.(\d+)\.self_attn_layer_norm\.', r'encoder.layers.\1.layer_norm.'),
    (r'^encoder\.layers\.(\d+)\.fc1\.', r'encoder.layers.\1.feed_forward.intermediate_dense.'),
    (r'^encoder\.layers\.(\d+)\.fc2\.', r'encoder.layers.\1.feed_forward.output_dense.'),
    (r'^encoder\.layers\.(\d+)\.final_layer_norm\.', r'encoder.layers.\1.final_layer_norm.'),
    (r'^encoder\.layer_norm\.', 'encoder.layer_norm.'),
]
_SSL_PREFIX = 'ssl_model.model.'


def convert_state_dict(sd, target_keys):
    """Rename an official SSL-AASIST (fairseq) state_dict onto SSLAASIST.
    Raises on any fairseq key it does not know how to map."""
    out, dropped = {}, []
    for k, v in sd.items():
        k = k[len('module.'):] if k.startswith('module.') else k
        if k.startswith(_SSL_PREFIX):
            s = k[len(_SSL_PREFIX):]
            if s.startswith(_FAIRSEQ_DROP):
                dropped.append(k)
                continue
            for pat, rep in _FAIRSEQ_RULES:
                if re.match(pat, s):
                    s = re.sub(pat, rep, s)
                    break
            else:
                raise KeyError(f'unmapped fairseq key: {k}')
            k = _SSL_PREFIX + s
            # weight-norm naming differs across torch versions
            if k.endswith('conv.weight_g') and k not in target_keys:
                k = k.replace('conv.weight_g', 'conv.parametrizations.weight.original0')
            elif k.endswith('conv.weight_v') and k not in target_keys:
                k = k.replace('conv.weight_v', 'conv.parametrizations.weight.original1')
        out[k] = v
    return out, dropped


def load_ssl_aasist_checkpoint(model, path, map_location='cpu'):
    sd = torch.load(path, map_location=map_location, weights_only=False)
    if isinstance(sd, dict) and 'model' in sd and not any(k.endswith('.weight') for k in sd):
        sd = sd['model']
    converted, dropped = convert_state_dict(sd, set(model.state_dict().keys()))
    model.load_state_dict(converted, strict=True)        # any gap raises
    return dropped


def build_ssl_aasist(checkpoint_path, device='cpu'):
    model = SSLAASIST()
    dropped = load_ssl_aasist_checkpoint(model, checkpoint_path)
    return model.to(device).eval(), dropped


def pad_fixed(x, max_len=NB_SAMP):
    """First `max_len` samples; tile-repeat shorter clips (upstream `pad`)."""
    n = x.shape[-1]
    if n == 0:
        return x.new_zeros(*x.shape[:-1], max_len)
    if n >= max_len:
        return x[..., :max_len]
    reps = max_len // n + 1
    return x.repeat(reps)[:max_len] if x.ndim == 1 else x.repeat(1, reps)[..., :max_len]

Writing /kaggle/working/ssl_aasist_net.py


In [3]:
%%writefile /kaggle/working/ssl_aasist_finetune.py
"""Fine-tuning helpers for SSL-AASIST (Phase 2e). Builds on ssl_aasist_net.py.

1. LayerWeightedSSL — the back-end reads a learnable weighted sum of ALL 25
   XLS-R hidden states instead of only the last one. Middle XLS-R layers carry
   more transferable spoofing cues (SLS, ACM MM 2024; probing-based layer
   selection, 2025). Weights start one-hot on the final layer, so at step 0 the
   model is numerically identical to the pretrained checkpoint (checked by
   `assert_equivalent`).
2. degrade — class-agnostic channel augmentation (narrowband 8 kHz, G.711
   mu-law, bandwidth limits, reverberation, additive noise, gain/clipping).
   Applied to bona fide AND spoof alike, so channel damage cannot become a
   label cue.
3. Trainable-only checkpoints (~2 MB instead of 1.27 GB) for resume + push.
"""
import math
import os
import subprocess

import numpy as np
import torch
import torch.nn as nn
import torchaudio.functional as AF

SR = 16000


# ---------------- layer-weighted front-end ----------------
class LayerWeightedSSL(nn.Module):
    """Drop-in replacement for ssl_aasist_net.SSLModel.

    Slot 24 is `last_hidden_state` (post final LayerNorm — what the pretrained
    back-end was trained on). Slots 0..23 are the earlier residual streams passed
    through the same frozen final LayerNorm so their scale matches slot 24.
    (In current transformers, `hidden_states[-1]` is NOT `last_hidden_state`.)
    """

    def __init__(self, base_ssl, unfreeze_top_k=0):
        super().__init__()
        self.model = base_ssl.model                       # keeps 'ssl_model.model.*' keys
        self.out_dim = base_ssl.out_dim
        n_layers = self.model.config.num_hidden_layers
        w = torch.zeros(n_layers + 1)
        w[-1] = 1.0
        self.layer_weights = nn.Parameter(w)
        self.unfreeze_top_k = unfreeze_top_k
        for p in self.model.parameters():
            p.requires_grad = False
        if unfreeze_top_k > 0:
            for layer in self.model.encoder.layers[-unfreeze_top_k:]:
                for p in layer.parameters():
                    p.requires_grad = True
            for p in self.model.encoder.layer_norm.parameters():
                p.requires_grad = True

    def extract_feat(self, x):
        if x.ndim == 3:
            x = x[:, :, 0]
        self.model.eval()                                  # no dropout/layerdrop in the encoder
        with torch.set_grad_enabled(torch.is_grad_enabled() and self.unfreeze_top_k > 0):
            out = self.model(x, output_hidden_states=True)
            ln = self.model.encoder.layer_norm
            stack = torch.stack([ln(h) for h in out.hidden_states[:-1]] + [out.last_hidden_state])
        return torch.einsum('l,lbtd->btd', self.layer_weights.to(stack.dtype), stack)


def attach_layer_weighting(model, unfreeze_top_k=0):
    model.ssl_model = LayerWeightedSSL(model.ssl_model, unfreeze_top_k)
    return model


@torch.no_grad()
def assert_equivalent(model_new, model_ref, x, atol=1e-4):
    """Layer-weighted model at init must reproduce the pretrained logits."""
    a, b = model_new.eval()(x), model_ref.eval()(x)
    diff = (a - b).abs().max().item()
    assert diff < atol, f'layer-weighted init differs from pretrained by {diff:.3e}'
    return diff


def layer_weight_report(model):
    w = model.ssl_model.layer_weights.detach().float().cpu().numpy()
    top = np.argsort(-np.abs(w))[:5]
    return 'top layers: ' + ', '.join(f'L{i}={w[i]:+.3f}' for i in top)


# ---------------- channel augmentation ----------------
def _rms(x):
    return x.pow(2).mean().sqrt().clamp_min(1e-8)


def _fit(y, n):
    return y[:n] if y.shape[-1] >= n else torch.nn.functional.pad(y, (0, n - y.shape[-1]))


# ---- real codec round-trips via an ffmpeg subprocess (lossy telephony/streaming) ----
FFMPEG = os.environ.get('FFMPEG_BIN', 'ffmpeg')
CODEC_SPECS = {
    # name: (ffmpeg encoder, ffmpeg muxer format, codec sample rate, bitrates or None)
    'g711_alaw': ('pcm_alaw', 'alaw', 8000, None),
    'gsm_fr':    ('libgsm', 'gsm', 8000, None),
    'amr_nb':    ('libopencore_amrnb', 'amr', 8000, [4750, 5900, 7400, 12200]),
    'amr_wb':    ('libvo_amrwbenc', 'amr', 16000, [6600, 12650, 23850]),
    'opus':      ('libopus', 'ogg', 16000, [6000, 12000, 24000]),
    'mp3':       ('libmp3lame', 'mp3', 16000, [32000, 64000]),
    'aac':       ('aac', 'adts', 16000, [32000, 64000]),
}
_codec_cache = None


def available_codecs():
    """CODEC_SPECS names whose encoder exists in the ffmpeg binary; [] if ffmpeg missing."""
    global _codec_cache
    if _codec_cache is None:
        try:
            out = subprocess.run([FFMPEG, '-hide_banner', '-encoders'],
                                 capture_output=True, text=True, timeout=30).stdout
            _codec_cache = [name for name, (enc, *_rest) in CODEC_SPECS.items() if enc in out]
        except Exception:
            _codec_cache = []
    return _codec_cache


def _ffmpeg_pipe(args, pcm):
    return subprocess.run(args, input=pcm, capture_output=True, timeout=60).stdout


def codec_roundtrip(x, name, rng):
    """float32 16 kHz mono tensor -> lossy codec encode+decode via ffmpeg pipes.
    Any failure returns x unchanged (a DataLoader worker must never crash)."""
    try:
        enc, fmt, codec_sr, bitrates = CODEC_SPECS[name]
        pcm = x.detach().cpu().numpy().astype('<f4').tobytes()
        cmd = [FFMPEG, '-hide_banner', '-loglevel', 'error',
               '-f', 'f32le', '-ar', str(SR), '-ac', '1', '-i', 'pipe:0',
               '-ar', str(codec_sr), '-ac', '1', '-c:a', enc]
        if bitrates:
            cmd += ['-b:a', str(bitrates[int(rng.integers(len(bitrates)))])]
        cmd += ['-f', fmt, 'pipe:1']
        blob = _ffmpeg_pipe(cmd, pcm)
        if not blob:
            return x
        tail = ['-i', 'pipe:0', '-f', 'f32le', '-ar', str(SR), '-ac', '1', 'pipe:1']
        dec = [FFMPEG, '-hide_banner', '-loglevel', 'error', '-f', fmt] + tail
        out = _ffmpeg_pipe(dec, blob)
        if not out:                                     # fmt may not be a demuxer name (e.g. adts)
            out = _ffmpeg_pipe([FFMPEG, '-hide_banner', '-loglevel', 'error'] + tail, blob)
        if not out:
            return x
        y = torch.from_numpy(np.frombuffer(out, dtype='<f4').copy())
        return _fit(y, x.shape[-1])
    except Exception:
        return x


def degrade(x, rng, codecs=None, p_codec=0.0):
    """x: 1-D float tensor @16 kHz. rng: np.random.Generator. Returns same length.
    With codecs falsy the RNG draw sequence is identical to the 2e version."""
    n = x.shape[-1]
    y = x.clone()
    if rng.random() < 0.4:                                  # bandwidth: phone / low-bitrate
        low_sr = int(rng.choice([8000, 8000, 11025, 12000]))
        y = _fit(AF.resample(AF.resample(y, SR, low_sr), low_sr, SR), n)
        if low_sr == 8000:
            y = AF.highpass_biquad(y, SR, 300.0)
            if rng.random() < 0.6:                          # G.711 mu-law
                peak = y.abs().max().clamp_min(1e-6)
                q = AF.mu_law_encoding((y / peak).clamp(-1, 1), 256)
                y = AF.mu_law_decoding(q, 256) * peak
    if rng.random() < 0.3:                                  # synthetic room reverb
        t60 = rng.uniform(0.2, 0.8)
        L = int(t60 * SR)
        t = torch.arange(L, dtype=y.dtype) / SR
        rir = torch.from_numpy(rng.standard_normal(L).astype(np.float32)) * torch.exp(-6.9 * t / t60)
        rir[0] = 1.0
        rir = rir / rir.norm()
        wet = AF.fftconvolve(y, rir)[:n]
        y = wet * (_rms(y) / _rms(wet))
    if rng.random() < 0.3:                                  # additive noise, 10-30 dB SNR
        noise = torch.from_numpy(rng.standard_normal(n).astype(np.float32))
        if rng.random() < 0.5:
            noise = AF.lowpass_biquad(noise, SR, float(rng.uniform(500, 4000)))
        snr_db = rng.uniform(10, 30)
        y = y + noise * (_rms(y) / _rms(noise)) / (10 ** (snr_db / 20))
    if codecs:                                              # real lossy codec round-trip
        if rng.random() < p_codec:
            y = codec_roundtrip(y, codecs[int(rng.integers(len(codecs)))], rng)
    if rng.random() < 0.5:                                  # level + clipping
        y = (y * 10 ** (rng.uniform(-12, 6) / 20)).clamp(-1, 1)
    return y


# ---------------- RADAR-style augmentation (Phase 2i) ----------------
def _tile_to(x, n):
    if x.shape[-1] >= n:
        return x[:n]
    return x.repeat(-(-n // x.shape[-1]))[:n]


def radar_degrade(x, rng, codecs=None, p_codec=0.0, music=None):
    """degrade() plus RADAR-2026-style media transforms (Phase 2i).

    degrade() itself is untouched so 2f/2g/2h runs keep identical RNG streams.
    Each extra stage is class-agnostic with its own draw; returns length n.
    """
    n = x.shape[-1]
    y = degrade(x, rng, codecs=codecs, p_codec=p_codec)
    if rng.random() < 0.2:                                  # fade-in / fade-out
        which = int(rng.integers(3))
        if which in (0, 2):
            k = int(rng.uniform(0.05, 0.5) * SR)
            y[:k] = y[:k] * torch.linspace(0, 1, k, dtype=y.dtype)
        if which in (1, 2):
            k = int(rng.uniform(0.05, 0.5) * SR)
            y[-k:] = y[-k:] * torch.linspace(1, 0, k, dtype=y.dtype)
    if rng.random() < 0.2:                                  # silence trim + tile
        fr = max(int(0.02 * SR), 1)
        m = y.shape[-1] - y.shape[-1] % fr
        rms = y[:m].view(-1, fr).pow(2).mean(-1).sqrt() if m else torch.zeros(0)
        if rms.numel():
            keep = (rms >= rms.max() * 10 ** (-40 / 20)).nonzero().flatten()
            a, b = (int(keep[0]) * fr, int(keep[-1] + 1) * fr) if len(keep) else (0, n)
            y = _tile_to(y[a:b] if b > a else y, n)
    if rng.random() < 0.2:                                  # zero pad + crop
        k = int(rng.uniform(0.1, 1.0) * SR)
        z = torch.zeros(k, dtype=y.dtype)
        y = torch.cat([z, y])[:n] if rng.random() < 0.5 else torch.cat([y, z])[:n]
    if rng.random() < 0.25:                                 # dynamic range compression
        fr = max(int(0.01 * SR), 1)
        m = y.shape[-1] - y.shape[-1] % fr
        if m:
            env = _tile_to(y[:m].view(-1, fr).pow(2).mean(-1).sqrt()
                           .repeat_interleave(fr), n).clamp_min(1e-8)
            t = 10 ** (rng.uniform(-30, -15) / 20)
            r = rng.uniform(2, 8)
            gain = torch.where(env > t, (env / t) ** (1.0 / r - 1.0),
                               torch.ones_like(env))
            y2 = y * gain
            y = y2 * (_rms(y) / _rms(y2))
    if rng.random() < 0.3:                                  # peak normalisation
        peak = y.abs().max().clamp_min(1e-6)
        y = y * (10 ** (rng.uniform(-6, -0.1) / 20) / peak)
    if music and rng.random() < 0.15:                       # background music mix
        mx = _tile_to(music[int(rng.integers(len(music)))].to(y.dtype), n)
        snr = rng.uniform(5, 20)
        y = y + mx * (_rms(y) / _rms(mx)) / (10 ** (snr / 20))
    return y


# ---------------- trainable-only checkpoints ----------------
def trainable_state(model):
    trainable = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu() for k, v in model.state_dict().items()
            if not k.startswith('ssl_model.model.') or k in trainable}


def load_trainable_state(model, sd):
    missing, unexpected = model.load_state_dict(sd, strict=False)
    bad_missing = [k for k in missing if not k.startswith('ssl_model.model.')]
    assert not bad_missing and not unexpected, f'checkpoint mismatch: {bad_missing[:5]} {unexpected[:5]}'

Writing /kaggle/working/ssl_aasist_finetune.py


In [4]:
%%writefile /kaggle/working/antispoof_metrics.py
"""Anti-spoofing metrics (ASVspoof 5 Track 1 conventions) + LLR calibration.

Scores: higher = more bona fide. Labels: 1 = bona fide, 0 = spoof.
Formulas follow asvspoof-challenge/asvspoof5 evaluation-package/calculate_modules.py:
  beta      = Cmiss * (1 - Pspoof) / (Cfa * Pspoof)            (= 1.90 for 1, 10, 0.05)
  minDCF    = min_t [Cmiss*(1-Pspoof)*Pmiss(t) + Cfa*Pspoof*Pfa(t)] / min(Cmiss*(1-Pspoof), Cfa*Pspoof)
            = min_t [beta * Pmiss(t) + Pfa(t)]
  actDCF    = same DCF at the Bayes threshold t = -log(beta), scores must be LLRs
  Cllr      = 0.5 * (mean log2(1 + e^-llr | bona) + mean log2(1 + e^llr | spoof))
"""
import numpy as np

PSPOOF, CMISS, CFA = 0.05, 1.0, 10.0
BETA = CMISS * (1 - PSPOOF) / (CFA * PSPOOF)
BAYES_THRESHOLD = -np.log(BETA)
_C_DEF = min(CMISS * (1 - PSPOOF), CFA * PSPOOF)


def _split(labels, scores):
    labels, scores = np.asarray(labels), np.asarray(scores, dtype=np.float64)
    return scores[labels == 1], scores[labels == 0]


def det_curve(bona, spoof):
    """Official compute_det_curve: frr/far over all thresholds (sorted scores)."""
    n = bona.size + spoof.size
    s = np.concatenate([bona, spoof])
    lab = np.concatenate([np.ones(bona.size), np.zeros(spoof.size)])
    idx = np.argsort(s, kind='mergesort')
    lab = lab[idx]
    tar_sums = np.cumsum(lab)
    non_sums = spoof.size - (np.arange(1, n + 1) - tar_sums)
    frr = np.concatenate([[0.0], tar_sums / bona.size])
    far = np.concatenate([[1.0], non_sums / spoof.size])
    thr = np.concatenate([[s[idx[0]] - 0.001], s[idx]])
    return frr, far, thr


def eer(labels, scores):
    bona, spoof = _split(labels, scores)
    frr, far, thr = det_curve(bona, spoof)
    i = int(np.argmin(np.abs(frr - far)))
    return float((frr[i] + far[i]) / 2), float(thr[i])


def min_dcf(labels, scores):
    bona, spoof = _split(labels, scores)
    frr, far, thr = det_curve(bona, spoof)
    c = (CMISS * (1 - PSPOOF) * frr + CFA * PSPOOF * far) / _C_DEF
    i = int(np.argmin(c))
    return float(c[i]), float(thr[i])


def rates_at(labels, scores, threshold):
    """(spoof FAR, bona-fide FRR) when accepting score >= threshold."""
    bona, spoof = _split(labels, scores)
    far = float(np.mean(spoof >= threshold)) if spoof.size else float('nan')
    frr = float(np.mean(bona < threshold)) if bona.size else float('nan')
    return far, frr


def act_dcf(labels, llrs, threshold=BAYES_THRESHOLD):
    far, frr = rates_at(labels, llrs, threshold)
    return float((CMISS * (1 - PSPOOF) * frr + CFA * PSPOOF * far) / _C_DEF)


def cllr(labels, llrs):
    bona, spoof = _split(labels, llrs)
    return float(0.5 * (np.mean(np.logaddexp(0, -bona)) + np.mean(np.logaddexp(0, spoof))) / np.log(2))


# ---------------- affine LLR calibration (prior-weighted logistic regression) ----------------
EFFECTIVE_PRIOR = BETA / (1 + BETA)   # bona-fide prior at which the ASVspoof 5 costs are equivalent


def fit_calibration(labels, scores, prior=EFFECTIVE_PRIOR, iters=50):
    """Fit llr = a*s + b minimising the prior-weighted logistic loss (BOSARIS-style).

    Newton's method on 2 parameters; scores are standardised internally for stability.
    Returns dict(a, b, prior) with llr = a * score + b.
    """
    bona, spoof = _split(labels, scores)
    mu, sd = float(np.mean(np.concatenate([bona, spoof]))), float(np.std(np.concatenate([bona, spoof])) + 1e-12)
    xb, xs = (bona - mu) / sd, (spoof - mu) / sd
    lo = np.log(prior / (1 - prior))
    w = np.array([1.0, 0.0])
    for _ in range(iters):
        g, H = np.zeros(2), np.zeros((2, 2))
        for x, sign, weight in ((xb, 1.0, prior / xb.size), (xs, -1.0, (1 - prior) / xs.size)):
            z = sign * (w[0] * x + w[1] + lo)
            p = 1.0 / (1.0 + np.exp(np.clip(z, -50, 50)))        # d/dz softplus(-z) = -p
            X = np.stack([x, np.ones_like(x)], 1)
            g += weight * (-(sign * p) @ X)
            H += weight * (X.T * (p * (1 - p))) @ X
        step = np.linalg.solve(H + 1e-9 * np.eye(2), g)
        w -= step
        if np.max(np.abs(step)) < 1e-10:
            break
    a = w[0] / sd
    return {'a': float(a), 'b': float(w[1] - a * mu), 'prior': float(prior)}


def apply_calibration(scores, cal):
    return cal['a'] * np.asarray(scores, dtype=np.float64) + cal['b']


def full_report(labels, scores, cal=None, fixed_threshold=None):
    """All Track-1 metrics for one score set. `cal` maps raw scores to LLRs."""
    out = {'n_bona': int(np.sum(np.asarray(labels) == 1)), 'n_spoof': int(np.sum(np.asarray(labels) == 0))}
    if out['n_bona'] and out['n_spoof']:
        out['eer'], out['eer_threshold'] = eer(labels, scores)
        out['min_dcf'], _ = min_dcf(labels, scores)
    if cal is not None:
        llr = apply_calibration(scores, cal)
        out['far_bayes'], out['frr_bayes'] = rates_at(labels, llr, BAYES_THRESHOLD)
        if out['n_bona'] and out['n_spoof']:
            out['act_dcf'] = act_dcf(labels, llr)
            out['cllr'] = cllr(labels, llr)
    if fixed_threshold is not None:
        out['far_fixed'], out['frr_fixed'] = rates_at(labels, scores, fixed_threshold)
    return out

Writing /kaggle/working/antispoof_metrics.py


In [5]:
# ============================================================
# 1. Models: ECAPA-TDNN (ASV, Phase 1) & SSL-AASIST (CM, Phase 2f)
# ============================================================
import importlib, ssl_aasist_net, ssl_aasist_finetune, antispoof_metrics
for m in (ssl_aasist_net, ssl_aasist_finetune, antispoof_metrics):
    importlib.reload(m)
from ssl_aasist_net import build_ssl_aasist, pad_fixed, NB_SAMP
from ssl_aasist_finetune import attach_layer_weighting, load_trainable_state
from antispoof_metrics import (eer as am_eer, min_dcf, act_dcf, cllr, rates_at,
                               fit_calibration, apply_calibration, full_report,
                               BETA, BAYES_THRESHOLD, EFFECTIVE_PRIOR)

print(f'ASVspoof 5 Track-1 constants: beta={BETA:.2f} | '
      f'bayes_threshold={BAYES_THRESHOLD:.3f} | eff_prior={EFFECTIVE_PRIOR:.3f}')

# --- CM gate config + checkpoint: newest attached phase wins (2h > 2g > 2f) ---
GATE, CM_STATE, CM_VARIANT, CM_CKPT = {}, None, None, None
for _variant, _ckpt_name, _gate_name in (('phase2j', 'ssl_2j_best.pt', 'phase2j_gate_config.json'),
                                         ('phase2i', 'ssl_2i_best.pt', 'phase2i_gate_config.json'),
                                         ('phase2h', 'ssl_2h_best.pt', 'phase2h_gate_config.json'),
                                         ('phase2g', 'ssl_2g_best.pt', 'phase2g_gate_config.json'),
                                         ('phase2f', 'ssl_2f_best.pt', 'phase2f_gate_config.json')):
    _ckpt = next(iter(glob.glob(f'/kaggle/input/**/{_ckpt_name}', recursive=True)), None)
    if not _ckpt:
        continue
    CM_STATE = torch.load(_ckpt, map_location='cpu', weights_only=False)['trainable']
    CM_VARIANT, CM_CKPT = _variant, _ckpt_name
    print(f'{_variant} checkpoint:', _ckpt)
    _gate = next(iter(glob.glob(f'/kaggle/input/**/{_gate_name}', recursive=True)), None)
    if _gate:
        with open(_gate) as f:
            GATE = json.load(f)
        print(f'{_variant} gate config:', _gate)
    break
if CM_STATE is None:
    print('no ssl_2j/2i/2h/2g/2f_best.pt under /kaggle/input -> '
          'CM scoring disabled (simulation harness still runs)')

# --- build the countermeasure model (optional: only needed for real audio scoring) ---
cm_model, CM_READY = None, False
if CM_STATE is not None:
    found = [p for name in ('LA_model.pth', 'best_SSL_model_LA.pth')
             for p in glob.glob(f'/kaggle/input/**/{name}', recursive=True)]
    base = found[0] if found else None
    if base is None:
        try:
            from huggingface_hub import hf_hub_download
            base = hf_hub_download('SpeechAntiSpoofingBenchmarks/W2V2-AASIST',
                                   'LA_model.pth', local_dir=WORK)
        except Exception as e:
            print('base SSL checkpoint unavailable:', e)
    if base:
        cm_model, _ = build_ssl_aasist(base, device)
        _k = int(GATE.get('unfreeze_top_k',
                          6 if CM_VARIANT in ('phase2j', 'phase2i', 'phase2h', 'phase2g') else 4))
        cm_model = attach_layer_weighting(cm_model, _k).to(device)
        load_trainable_state(cm_model, CM_STATE)
        cm_model.eval()
        CM_READY = True
        print(f'SSL-AASIST ({CM_VARIANT}, unfreeze_top_k={_k}) ready on', device)

# --- ECAPA-TDNN speaker verifier (Phase 1 convention: SpeakerRecognition) ---
asv_model, ASV_READY = None, False
try:
    from speechbrain.inference.speaker import SpeakerRecognition
    asv_model = SpeakerRecognition.from_hparams(
        source='speechbrain/spkrec-ecapa-voxceleb',
        savedir='/tmp/speechbrain_ecapa',
        run_opts={'device': str(device)})
    asv_model.eval()
    ASV_READY = True
    print('ECAPA-TDNN ready on', device)
except Exception as e:
    print('ECAPA unavailable ->', e)

@torch.no_grad()
def score_trial(enroll_wav, test_wav):
    '''enroll/test waveforms [1, samples] @16k -> (s_asv cosine, s_cm bona-fide logit).'''
    e1 = asv_model.encode_batch(enroll_wav.to(device)).squeeze()
    e2 = asv_model.encode_batch(test_wav.to(device)).squeeze()
    s_asv = F.cosine_similarity(e1.view(1, -1), e2.view(1, -1)).item()
    x = pad_fixed(test_wav.squeeze(0).to(device))
    s_cm = cm_model(x.unsqueeze(0))[:, 1].item()
    return float(s_asv), float(s_cm)

ASVspoof 5 Track-1 constants: beta=1.90 | bayes_threshold=-0.642 | eff_prior=0.655
phase2j checkpoint: /kaggle/input/datasets/avinnn/ssl-aasist-phase2j/ssl_2j_best.pt
phase2j gate config: /kaggle/input/datasets/avinnn/ssl-aasist-phase2j/phase2j_gate_config.json


LA_model.pth: reconstructing file:   0%|          |  0.00B / 1.27GB            

LA_model.pth: downloading bytes:           |  0.00B            

SSL-AASIST (phase2j, unfreeze_top_k=6) ready on cuda


hyperparams.yaml:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

embedding_model.ckpt: reconstructing file:   0%|          |  0.00B / 83.3MB            

... (3 outputs left out)

classifier.ckpt: downloading bytes:           |  0.00B            

label_encoder.txt:   0%|          | 0.00/129k [00:00<?, ?B/s]

Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


ECAPA-TDNN ready on cuda


In [6]:
# ============================================================
# 2. Metric engine: Level 1 (Track 1) + Level 2 SASV (Track 2)
# ============================================================
# Level 1 uses the shared antispoof_metrics engine (identical to Phase 2f):
#   am_eer(labels, scores) / min_dcf / act_dcf / cllr / rates_at / full_report
# Level 2 adds the ASVspoof-5-Track-2 architecture-agnostic DCF over trial triplets.

ALPHA, GAMMA = 1.58, 0.84          # SASV a-DCF cost parameters (roadmap §3.2)

def compute_actual_a_dcf(sasv_scores, trial_types, tau_fixed=0.0,
                         alpha=ALPHA, gamma=GAMMA):
    '''Actual a-DCF at the fixed deployed threshold tau_fixed (no hindsight).'''
    accept = np.asarray(sasv_scores) >= tau_fixed
    tar = np.asarray(trial_types) == 'target'
    non = np.asarray(trial_types) == 'nontarget'
    spf = np.asarray(trial_types) == 'spoof'
    p_miss = float(np.mean(~accept[tar]))
    p_fa_non = float(np.mean(accept[non]))
    p_fa_spf = float(np.mean(accept[spf]))
    return {'actual_a_DCF': float(alpha * p_miss + (1.0 - gamma) * p_fa_non + gamma * p_fa_spf),
            'P_miss_target': p_miss, 'P_fa_nontarget': p_fa_non, 'P_fa_spoof': p_fa_spf}

def compute_min_a_dcf(sasv_scores, trial_types, alpha=ALPHA, gamma=GAMMA):
    '''Minimum a-DCF over all candidate thresholds (oracle discrimination bound).'''
    s = np.asarray(sasv_scores, dtype=np.float64)
    t = np.asarray(trial_types)
    tar_s = np.sort(s[t == 'target'])
    non_s = np.sort(s[t == 'nontarget'])
    spf_s = np.sort(s[t == 'spoof'])
    cands = np.sort(np.unique(s))
    # Pmiss(t) = frac target < t ; Pfa(t) = frac non/spoof >= t  (searchsorted, O(u log n))
    pmiss = np.searchsorted(tar_s, cands, 'left') / tar_s.size
    pfa_n = 1.0 - np.searchsorted(non_s, cands, 'left') / non_s.size
    pfa_s = 1.0 - np.searchsorted(spf_s, cands, 'left') / spf_s.size
    cost = alpha * pmiss + (1.0 - gamma) * pfa_n + gamma * pfa_s
    i = int(np.argmin(cost))
    return {'min_a_DCF': float(cost[i]), 'min_a_DCF_threshold': float(cands[i])}

def compute_t_eer(sasv_scores, trial_types):
    '''Tandem EER: pooled EER of the joint score (target = accept, else reject).'''
    labels = (np.asarray(trial_types) == 'target').astype(int)
    return am_eer(labels, sasv_scores)[0]

print('SASV metric engine ready (alpha=%.2f, gamma=%.2f).' % (ALPHA, GAMMA))

SASV metric engine ready (alpha=1.58, gamma=0.84).


In [7]:
# ============================================================
# 3. LLR calibration engine (shared prior-weighted fit from Phase 2f)
# ============================================================

class LLRCalibrator:
    '''Affine LLR calibrator (llr = a*score + b) over antispoof_metrics.fit_calibration.

    Uses the identical Newton solver and effective prior beta/(1+beta) as Phase 2f,
    so calibrated LLRs are evaluated at the same Bayes threshold -log(beta) = -0.642.
    '''
    def __init__(self, prior=EFFECTIVE_PRIOR):
        self.prior = prior
        self.cal = {'a': 1.0, 'b': 0.0, 'prior': prior}

    def fit(self, scores, labels):
        self.cal = fit_calibration(np.asarray(labels), np.asarray(scores), prior=self.prior)
        print(f'  [LLR calibration] a={self.cal["a"]:.4f}  b={self.cal["b"]:.4f}  '
              f'(effective prior {self.cal["prior"]:.3f})')

    def transform(self, raw_scores):
        return apply_calibration(raw_scores, self.cal)

    @staticmethod
    def bayes_threshold():
        return BAYES_THRESHOLD

print('LLR calibrator ready (shared antispoof_metrics backend).')

LLR calibrator ready (shared antispoof_metrics backend).


In [8]:
# ============================================================
# 4. Joint SASV score-fusion architectures (roadmap section 5.1)
# ============================================================

class LinearLLRFusion:
    '''Strategy 1: linear LLR fusion  s = w1*llr_asv + w2*llr_cm + b.'''
    def __init__(self):
        self.clf = LogisticRegression(C=1e5)
        self.w1 = self.w2 = self.b = None

    def fit(self, llr_asv, llr_cm, joint_labels):
        self.clf.fit(np.column_stack([llr_asv, llr_cm]), joint_labels)
        self.w1, self.w2 = map(float, self.clf.coef_[0])
        self.b = float(self.clf.intercept_[0])
        print(f'  [linear fusion] w_asv={self.w1:.4f}  w_cm={self.w2:.4f}  b={self.b:.4f}')

    def predict_score(self, llr_asv, llr_cm):
        return self.w1 * llr_asv + self.w2 * llr_cm + self.b

class CascadedMinGate:
    '''Strategy 2: failsafe min-score gate  s = min(llr_asv - t_asv, llr_cm - t_cm).
    Accept only if s >= 0: either gate failing rejects the caller.'''
    def __init__(self, tau_asv=BAYES_THRESHOLD, tau_cm=BAYES_THRESHOLD):
        self.tau_asv, self.tau_cm = float(tau_asv), float(tau_cm)

    def predict_score(self, llr_asv, llr_cm):
        return np.minimum(llr_asv - self.tau_asv, llr_cm - self.tau_cm)

class MLPFusionHead(nn.Module):
    '''Strategy 3: 2-layer MLP over [llr_asv, llr_cm]; output logit = SASV score.'''
    def __init__(self, in_dim=2, hidden=16):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, 1))

    def forward(self, x):
        return self.net(x).squeeze(-1)

    def fit(self, llr_asv, llr_cm, joint_labels, epochs=400, lr=1e-2):
        X = torch.tensor(np.column_stack([llr_asv, llr_cm]), dtype=torch.float32)
        y = torch.tensor(np.asarray(joint_labels), dtype=torch.float32)
        opt = torch.optim.Adam(self.parameters(), lr=lr)
        lossf = nn.BCEWithLogitsLoss()
        self.train()
        for ep in range(epochs):
            opt.zero_grad()
            loss = lossf(self.forward(X), y)
            loss.backward()
            opt.step()
        self.eval()
        print(f'  [MLP fusion] trained {epochs} epochs | final BCE {loss.item():.4f}')

    def predict_score(self, llr_asv, llr_cm):
        X = torch.tensor(np.column_stack([llr_asv, llr_cm]), dtype=torch.float32)
        with torch.no_grad():
            return self.forward(X).numpy()

print('Score-fusion architectures loaded (linear / min-gate / MLP).')

Score-fusion architectures loaded (linear / min-gate / MLP).


In [9]:
# ============================================================
# 4b. Generate sasv_trials.csv from attached datasets (optional)
# ============================================================
# If no sasv_trials.csv is attached, build one from whatever is under
# /kaggle/input and save it to /kaggle/working (download it later to re-use
# as a pinned dataset input). Trial semantics (roadmap section 3.1):
#   target    -> same speaker, genuine voice      (enroll + test both bona fide)
#   nontarget -> different speaker, genuine voice (impostor)
#   spoof     -> enrolled speaker, synthetic voice
# Sources used when found:
#   In-the-Wild   meta.csv (file,speaker,label)          -> target/nontarget/spoof
#   VoxCeleb1     wav/idXXXXX/<yt>/NNNNN.wav             -> target/nontarget
#   ASVspoof 2019 LA cm protocols (dev -> val, eval -> eval) -> nontarget/spoof
import re, collections

SASV_CSV = os.path.join(WORK, 'sasv_trials.csv')
_attached = next(iter(glob.glob('/kaggle/input/**/sasv_trials.csv', recursive=True)), None)

TRIALS_PER_SPK = {'itw': 12, 'vox': 15, 'asv19_dev': 30, 'asv19_eval': 20}   # per speaker, per trial type
VAL_SPK_FRAC = 0.4        # In-the-Wild / VoxCeleb: a speaker goes wholly to val OR eval, never both
FIELDS = ['split', 'enroll', 'test', 'type', 'speaker', 'source']

def _spk_split(spks, seed):
    s = sorted(spks); random.Random(seed).shuffle(s)
    return set(s[:max(1, int(len(s) * VAL_SPK_FRAC))])

if _attached:
    print('attached sasv_trials.csv found ->', _attached)
else:
    rows, seen = [], set()
    def _add(split, e, t, y, spk, src):
        if (e, t, y) in seen:
            return
        seen.add((e, t, y))
        rows.append({'split': split, 'enroll': e, 'test': t, 'type': y, 'speaker': spk, 'source': src})

    # ---- source 1: In-the-Wild (speaker-labeled bona-fide + spoof) ----
    for mp in [p for p in glob.glob('/kaggle/input/**/meta.csv', recursive=True)
               if 'wild' in p.lower()][:1]:
        root = os.path.dirname(mp)
        by_spk = collections.defaultdict(lambda: {'bona': [], 'spoof': []})
        with open(mp) as f:
            for r in csv.DictReader(f):
                k = 'bona' if 'bona' in r.get('label', '').lower() else 'spoof'
                by_spk[r.get('speaker', '')][k].append(os.path.join(root, r['file']))
        spks = sorted(s for s in by_spk
                      if len(by_spk[s]['bona']) >= 2 and by_spk[s]['spoof'])
        val_set, rng = _spk_split(spks, 42), random.Random(42)
        for s in spks:
            sp, b, f = ('val' if s in val_set else 'eval'), by_spk[s]['bona'], by_spk[s]['spoof']
            others = [o for o in spks if o != s]
            for _ in range(TRIALS_PER_SPK['itw']):
                e, t = rng.sample(b, 2); _add(sp, e, t, 'target', s, 'itw')
                if others:
                    _add(sp, rng.choice(b), rng.choice(by_spk[rng.choice(others)]['bona']),
                         'nontarget', s, 'itw')
                _add(sp, rng.choice(b), rng.choice(f), 'spoof', s, 'itw')
        print(f'ITW     : {len(spks)} usable speakers ({len(val_set)} val / {len(spks) - len(val_set)} eval), '
              f'{os.path.dirname(root)}')

    # ---- source 2: VoxCeleb1 test (40 speakers, wav/idXXXXX/...) -> target/nontarget ----
    vox = collections.defaultdict(dict)
    for p in glob.glob('/kaggle/input/**/id10*/*/0*.wav', recursive=True) + \
             glob.glob('/kaggle/input/**/id10*/*/*.wav', recursive=True):
        m = re.search(r'/wav/(id\d+)/([^/]+)/([^/]+\.wav)$', p.replace('\\', '/'))
        if m:
            vox[m.group(1)][m.group(3) + '|' + m.group(2)] = p      # dedup repeated copies
    spks = sorted(s for s in vox if len(vox[s]) >= 2)
    if spks:
        val_set, rng = _spk_split(spks, 7), random.Random(7)
        for s in spks:
            sp, files = ('val' if s in val_set else 'eval'), sorted(vox[s])
            others = [o for o in spks if o != s]
            for _ in range(TRIALS_PER_SPK['vox']):
                e, t = rng.sample(files, 2); _add(sp, vox[s][e], vox[s][t], 'target', s, 'vox')
                if others:
                    o = vox[rng.choice(others)]
                    _add(sp, vox[s][rng.choice(files)], o[rng.choice(sorted(o))], 'nontarget', s, 'vox')
        print(f'VoxCeleb: {len(spks)} speakers ({len(val_set)} val / {len(spks) - len(val_set)} eval)')

    # ---- source 3: ASVspoof 2019 LA protocols (dev -> val, eval -> eval; speakers disjoint) ----
    for proto, tag in (('/kaggle/input/**/ASVspoof2019.LA.cm.dev.trl.txt', 'dev'),
                       ('/kaggle/input/**/ASVspoof2019.LA.cm.eval.trl.txt', 'eval')):
        pp = next(iter(glob.glob(proto, recursive=True)), None)
        if not pp:
            continue
        bon, spf = collections.defaultdict(list), collections.defaultdict(list)
        with open(pp) as f:
            for line in f:
                p = line.split()
                if len(p) >= 4:
                    (bon if p[-1] == 'bonafide' else spf)[p[0]].append(p[1] + '.flac')
        spks = sorted(s for s in bon if spf.get(s) and len(bon[s]) >= 2)
        rng, sp = random.Random(11), ('val' if tag == 'dev' else 'eval')
        for s in spks:
            others = [o for o in spks if o != s]
            for _ in range(TRIALS_PER_SPK['asv19_' + tag]):
                e, t = rng.sample(bon[s], 2); _add(sp, e, t, 'target', s, 'asv19_' + tag)
                if others:
                    _add(sp, rng.choice(bon[s]), rng.choice(bon[rng.choice(others)]),
                         'nontarget', s, 'asv19_' + tag)
                _add(sp, rng.choice(bon[s]), rng.choice(spf[s]), 'spoof', s, 'asv19_' + tag)
        print(f'ASV19 {tag}: {len(spks)} usable speakers')

    if rows:
        with open(SASV_CSV, 'w', newline='') as f:
            w = csv.DictWriter(f, fieldnames=FIELDS)
            w.writeheader(); w.writerows(rows)
        print(f'wrote {SASV_CSV}: {len(rows)} trials')
        for sp in ('val', 'eval'):
            c = collections.Counter(r['type'] for r in rows if r['split'] == sp)
            ns_ = len({r['speaker'] for r in rows if r['split'] == sp})
            print(f'   {sp:4s}: {dict(c)} | {ns_} speakers')
        both = ({r['speaker'] for r in rows if r['split'] == 'val'} &
                {r['speaker'] for r in rows if r['split'] == 'eval'})
        print(f'   speakers in BOTH val and eval: {len(both)}  (must be 0 for the calibration to be honest)')
    else:
        print('no usable audio datasets found -> simulated score triplets')

ITW     : 54 usable speakers (21 val / 33 eval), /kaggle/input/datasets/abdallamohamed312
VoxCeleb: 40 speakers (16 val / 24 eval)
ASV19 dev: 10 usable speakers
ASV19 eval: 48 usable speakers
wrote /kaggle/working/sasv_trials.csv: 6901 trials
   val : {'target': 784, 'nontarget': 792, 'spoof': 547} | 47 speakers
   eval: {'target': 1710, 'nontarget': 1716, 'spoof': 1352} | 105 speakers
   speakers in BOTH val and eval: 0  (must be 0 for the calibration to be honest)


In [10]:
# ============================================================
# 5. SASV trial harness: real CSV scoring if attached, else simulated
# ============================================================
# Real path: sasv_trials.csv with columns
#   split,enroll,test,type   (split in {val, eval}; type in {target,nontarget,spoof})
# attached as a dataset input OR generated into /kaggle/working by the cell
# above. Audio resolved under /kaggle/input (full path > suffix > basename).
# Requires CM_READY and ASV_READY.

def load_wav_16k(path):
    x, sr = torchaudio.load(path)
    if x.shape[0] > 1:
        x = x.mean(dim=0, keepdim=True)
    if sr != 16000:
        x = torchaudio.transforms.Resample(sr, 16000)(x)
    return x

_FLAT = None
def _resolve(p):
    '''CSV path -> real file: exact path > suffix match > basename (basenames are
    NOT unique across VoxCeleb speakers, so prefer full/relative paths in the CSV).'''
    global _FLAT
    if os.path.isfile(p):
        return p
    if _FLAT is None:
        _FLAT = (glob.glob('/kaggle/input/**/*.wav', recursive=True) +
                 glob.glob('/kaggle/input/**/*.flac', recursive=True) +
                 glob.glob(WORK + '/**/*.wav', recursive=True))
    norm = '/' + p.replace('\\', '/').strip('/')
    hit = next((f for f in _FLAT if f.replace('\\', '/').endswith(norm)), None)
    if hit is None:
        b = os.path.basename(p)
        hit = next((f for f in _FLAT if os.path.basename(f) == b), None)
    return hit

_E_CACHE, _C_CACHE = {}, {}

@torch.no_grad()
def _emb(p):
    if p not in _E_CACHE:
        _E_CACHE[p] = asv_model.encode_batch(load_wav_16k(p).to(device)).squeeze().view(1, -1)
    return _E_CACHE[p]

@torch.no_grad()
def _cm(p):
    if p not in _C_CACHE:
        x = pad_fixed(load_wav_16k(p).squeeze(0).to(device))
        _C_CACHE[p] = float(cm_model(x.unsqueeze(0))[:, 1].item())
    return _C_CACHE[p]

def score_trials_csv(csv_path):
    '''Score a sasv_trials.csv through ECAPA + SSL-AASIST. Embeddings / CM scores are cached
    per file (enrolment files repeat across trials). Optional columns: speaker, source.'''
    with open(csv_path) as f:
        rows = list(csv.DictReader(f))
    keys = ('asv', 'cm', 'type', 'spk', 'src')
    out = {k: {x: [] for x in keys} for k in ('val', 'eval')}
    missing = 0
    for i, r in enumerate(rows):
        ep, tp = _resolve(r['enroll']), _resolve(r['test'])
        if not ep or not tp:
            missing += 1
            continue
        k = 'val' if r.get('split', 'eval') == 'val' else 'eval'
        out[k]['asv'].append(float(F.cosine_similarity(_emb(ep), _emb(tp)).item()))
        out[k]['cm'].append(_cm(tp))
        out[k]['type'].append(r['type'])
        out[k]['spk'].append(r.get('speaker') or f'row{i}')
        out[k]['src'].append(r.get('source') or 'unknown')
        if (i + 1) % 500 == 0:
            print(f'  scored {i + 1}/{len(rows)} trials | {len(_E_CACHE)} files embedded')
    if missing:
        print(f'  WARNING: {missing} trials skipped (audio not found)')
    for k in out:
        out[k] = (np.array(out[k]['asv']), np.array(out[k]['cm']), np.array(out[k]['type']),
                  np.array(out[k]['spk']), np.array(out[k]['src']))
    return out

def generate_simulated_sasv_trials(n_targets=1500, n_nontargets=1500, n_spoofs=1500):
    '''Score triplets matched to Phase 1 & Phase 2f observed distributions:
    ASV cosine — target ~0.75, impostor ~0.15, spoof-clone ~0.65 (deepfakes pass ECAPA);
    CM bona-fide logit — genuine ~+2.4, spoof ~-1.5 (2f-calibrated LLR regime).'''
    asv = np.concatenate([np.random.normal(0.75, 0.10, n_targets),
                          np.random.normal(0.15, 0.12, n_nontargets),
                          np.random.normal(0.65, 0.15, n_spoofs)])
    cm = np.concatenate([np.random.normal(2.50, 0.80, n_targets),
                         np.random.normal(2.40, 0.85, n_nontargets),
                         np.random.normal(-1.50, 1.10, n_spoofs)])
    types = np.array(['target'] * n_targets + ['nontarget'] * n_nontargets +
                     ['spoof'] * n_spoofs)
    labels = np.array([1] * n_targets + [0] * (n_nontargets + n_spoofs))
    return asv, cm, types, labels

TRIALS_CSV = (next(iter(glob.glob('/kaggle/input/**/sasv_trials.csv', recursive=True)), None)
              or (SASV_CSV if os.path.isfile(SASV_CSV) else None))
REAL = TRIALS_CSV is not None and CM_READY and ASV_READY
print('sasv_trials.csv:', TRIALS_CSV or 'not attached',
      '| real scoring:', 'ON' if REAL else 'OFF -> simulated score triplets')

sasv_trials.csv: /kaggle/working/sasv_trials.csv | real scoring: ON


In [11]:
# ============================================================
# 6. Fit calibration + fusion, evaluate on held-out SASV trials
# ============================================================

if REAL:
    scored = score_trials_csv(TRIALS_CSV)
    val_asv, val_cm, val_types, val_spk, val_src = scored['val']
    eval_asv, eval_cm, eval_types, eval_spk, eval_src = scored['eval']
else:
    val_asv, val_cm, val_types, _ = generate_simulated_sasv_trials(1500, 1500, 1500)
    eval_asv, eval_cm, eval_types, _ = generate_simulated_sasv_trials(2000, 2000, 2000)
    val_spk, eval_spk = np.arange(len(val_types)), np.arange(len(eval_types))
    val_src = eval_src = np.array(['simulated'] * len(eval_types))

val_labels = (val_types == 'target').astype(int)      # joint accept = target only
eval_labels = (eval_types == 'target').astype(int)

# --- 1. fit both LLR calibrators on the pooled validation split ---
cal_asv = LLRCalibrator()
cal_asv.fit(val_asv, (val_types == 'target').astype(int))
cal_cm = LLRCalibrator()
cal_cm.fit(val_cm, (val_types != 'spoof').astype(int))   # CM positive = any bona-fide voice

val_llr_asv, val_llr_cm = cal_asv.transform(val_asv), cal_cm.transform(val_cm)
eval_llr_asv, eval_llr_cm = cal_asv.transform(eval_asv), cal_cm.transform(eval_cm)

# --- Level 1 report: standalone CM on calibrated LLRs (identical to Phase 2f) ---
cm_eval_labels = (eval_types != 'spoof').astype(int)
l1 = full_report(cm_eval_labels, eval_llr_cm, cal={'a': 1.0, 'b': 0.0})
print('\n================ LEVEL 1: STANDALONE CM (Track 1) ================')
print(f'  EER {l1["eer"]:.2%} | minDCF {l1["min_dcf"]:.3f} | '
      f'actDCF {l1["act_dcf"]:.3f} | Cllr {l1["cllr"]:.3f}')
print(f'  at Bayes threshold {BAYES_THRESHOLD:.3f}: '
      f'spoof FAR {l1["far_bayes"]:.2%} | bona FRR {l1["frr_bayes"]:.2%}')

# --- 2. fit fusion strategies on the same pooled validation split ---
linear_fusion = LinearLLRFusion()
linear_fusion.fit(val_llr_asv, val_llr_cm, val_labels)
cascaded_gate = CascadedMinGate(tau_asv=BAYES_THRESHOLD, tau_cm=BAYES_THRESHOLD)
mlp_fusion = MLPFusionHead()
mlp_fusion.fit(val_llr_asv, val_llr_cm, val_labels)

# --- Level 2 evaluation on held-out trials ---
strategies = {
    'Standalone ASV (no gate)': eval_llr_asv,
    'Standalone CM (no ASV)': eval_llr_cm,
    'S1: linear LLR fusion': linear_fusion.predict_score(eval_llr_asv, eval_llr_cm),
    'S2: cascaded min-gate': cascaded_gate.predict_score(eval_llr_asv, eval_llr_cm),
    'S3: MLP fusion head': mlp_fusion.predict_score(eval_llr_asv, eval_llr_cm),
}

print('\n================ LEVEL 2: JOINT SASV (Track 2, held-out) ================')
results = []
for name, s in strategies.items():
    r = compute_actual_a_dcf(s, eval_types, tau_fixed=0.0)
    r.update(compute_min_a_dcf(s, eval_types))
    r['t_EER'] = compute_t_eer(s, eval_types)
    r['Strategy'] = name
    results.append(r)
    print(f"\n[{name}]")
    print(f"  actual a-DCF {r['actual_a_DCF']:.4f} | min a-DCF {r['min_a_DCF']:.4f} "
          f"(tau* {r['min_a_DCF_threshold']:+.2f}) | t-EER {r['t_EER']:.2%}")
    print(f"  target miss {r['P_miss_target']:.2%} | impostor FA {r['P_fa_nontarget']:.2%} "
          f"| spoof FA {r['P_fa_spoof']:.2%}")

# --- how many trials stand behind these numbers, and how uncertain are they? ---
print('\n---- held-out evidence ----')
for src in sorted(set(eval_src)):
    m = eval_src == src
    c = collections.Counter(eval_types[m])
    print(f'  {src:12s}: {dict(c)} | {len(set(eval_spk[m]))} speakers')
print(f'  TOTAL eval: {dict(collections.Counter(eval_types))} | {len(set(eval_spk))} speakers '
      f'| val: {dict(collections.Counter(val_types))} | {len(set(val_spk))} speakers')

def speaker_bootstrap(strat, types, spk, B=300, seed=0):
    '''Resample whole speakers (trials of one speaker are correlated) -> 90% CIs.'''
    types, spk = np.asarray(types), np.asarray(spk)
    uniq = np.unique(spk)
    by = {u: np.where(spk == u)[0] for u in uniq}
    rng = np.random.default_rng(seed)
    acc = {k: [] for k in strat}
    for _ in range(B):
        ix = np.concatenate([by[u] for u in rng.choice(uniq, len(uniq))])
        if any(not (types[ix] == t).any() for t in ('target', 'nontarget', 'spoof')):
            continue
        for k, sc in strat.items():
            r = compute_actual_a_dcf(sc[ix], types[ix], tau_fixed=0.0)
            acc[k].append((r['actual_a_DCF'], r['P_miss_target'], r['P_fa_nontarget'], r['P_fa_spoof']))
    return {k: np.percentile(np.array(v), [5, 50, 95], axis=0) for k, v in acc.items() if v}

BOOT3 = speaker_bootstrap(strategies, eval_types, eval_spk)
print('\n---- 90% speaker-bootstrap intervals (actual a-DCF | target miss | impostor FA | spoof FA) ----')
for k, q in BOOT3.items():
    print(f'  {k:26s} a-DCF {q[1][0]:.3f} [{q[0][0]:.3f}, {q[2][0]:.3f}] | miss {q[1][1]:.1%} [{q[0][1]:.1%}, {q[2][1]:.1%}] '
          f'| impFA {q[1][2]:.1%} [{q[0][2]:.1%}, {q[2][2]:.1%}] | spoofFA {q[1][3]:.1%} [{q[0][3]:.1%}, {q[2][3]:.1%}]')

  scored 500/6901 trials | 865 files embedded
  scored 1000/6901 trials | 1579 files embedded
  scored 1500/6901 trials | 2233 files embedded
  scored 2000/6901 trials | 2896 files embedded
  scored 2500/6901 trials | 3649 files embedded
  scored 3000/6901 trials | 4385 files embedded
  scored 3500/6901 trials | 5112 files embedded
  scored 4000/6901 trials | 5779 files embedded
  scored 4500/6901 trials | 6530 files embedded
  scored 5000/6901 trials | 7281 files embedded
  scored 5500/6901 trials | 7992 files embedded
  scored 6000/6901 trials | 8686 files embedded
  scored 6500/6901 trials | 9308 files embedded
  [LLR calibration] a=10.1880  b=-3.9875  (effective prior 0.655)
  [LLR calibration] a=2.2655  b=0.0549  (effective prior 0.655)

================ LEVEL 1: STANDALONE CM (Track 1) ================
  EER 6.59% | minDCF 0.158 | actDCF 0.181 | Cllr 0.256
  at Bayes threshold -0.642: spoof FAR 14.87% | bona FRR 1.69%
  [linear fusion] w_asv=1.4877  w_cm=0.4412  b=-1.3357
  [MLP 

In [12]:
# ============================================================
# 7. Export Phase 3 configuration & push to Kaggle
# ============================================================

OUT_DIR = os.path.join(WORK, 'phase3_out')
os.makedirs(OUT_DIR, exist_ok=True)

config = {
    'phase': 'Phase 3 joint SASV integration',
    'asv_model': 'speechbrain/spkrec-ecapa-voxceleb',
    'cm_model': {'variant': CM_VARIANT, 'checkpoint': f'{CM_CKPT}["trainable"]',
                 'gate_config': bool(GATE), 'unfreeze_top_k': GATE.get('unfreeze_top_k', 4)},
    'llr_asv': {'a': cal_asv.cal['a'], 'b': cal_asv.cal['b'], 'prior': cal_asv.cal['prior']},
    'llr_cm': {'a': cal_cm.cal['a'], 'b': cal_cm.cal['b'], 'prior': cal_cm.cal['prior']},
    'fusion': {
        'linear': {'w_asv': linear_fusion.w1, 'w_cm': linear_fusion.w2, 'b': linear_fusion.b},
        'cascaded_min_gate': {'tau_asv': cascaded_gate.tau_asv, 'tau_cm': cascaded_gate.tau_cm},
        'mlp_state_dict': 'phase3_mlp_fusion.pt',
    },
    'beta': BETA,
    'bayes_threshold': BAYES_THRESHOLD,
    'a_dcf_costs': {'alpha': ALPHA, 'gamma': GAMMA},
    'joint_threshold': 0.0,
    'score_semantics': 'sasv score >= 0 -> accept caller (target AND bona fide)',
    'trial_source': 'real sasv_trials.csv' if REAL else 'simulated (Phase 1/2f distributions)',
    'trial_design': {'per_speaker': TRIALS_PER_SPK if 'TRIALS_PER_SPK' in dir() else None,
                     'val_speaker_fraction': VAL_SPK_FRAC if 'VAL_SPK_FRAC' in dir() else None,
                     'n_eval': {t: int((eval_types == t).sum()) for t in ('target', 'nontarget', 'spoof')},
                     'n_eval_speakers': int(len(set(eval_spk)))},
    'speaker_bootstrap_90': {k: {'a_dcf': [float(x) for x in q[:, 0]], 'spoof_fa': [float(x) for x in q[:, 3]]}
                             for k, q in BOOT3.items()},
    'evaluation_summary': results,
    'level1_cm': l1,
}
with open(os.path.join(OUT_DIR, 'phase3_sasv_config.json'), 'w') as f:
    json.dump(config, f, indent=2)

torch.save({'state_dict': mlp_fusion.state_dict(), 'in_dim': 2, 'hidden': 16},
           os.path.join(OUT_DIR, 'phase3_mlp_fusion.pt'))

cols = ['Strategy', 'actual_a_DCF', 'min_a_DCF', 'min_a_DCF_threshold',
        't_EER', 'P_miss_target', 'P_fa_nontarget', 'P_fa_spoof']
with open(os.path.join(OUT_DIR, 'sasv_bench_results.csv'), 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    for r in results:
        w.writerow({k: r[k] for k in cols})

print('Exported Phase 3 artifacts to', OUT_DIR)
push_dir(OUT_DIR, 'ssl-aasist-phase3-sasv', 'Phase 3 joint SASV integration',
         'Phase 3 SASV score fusion and gate config')

Exported Phase 3 artifacts to /kaggle/working/phase3_out
   pushed -> avinnn/ssl-aasist-phase3-sasv


In [13]:
# ============================================================
# 8. Inference: verify a caller against an enrolled speaker
# ============================================================
# Same session: uses the in-memory models, calibrators and fusion weights.
# Fresh kernel: attach avinnn/ssl-aasist-phase3-sasv + ssl-aasist-phase2f as
# inputs and re-run cells 0-8; this block rebuilds fusion weights from the
# pushed config when they are not already in memory.

_cfg = os.path.join(OUT_DIR, 'phase3_sasv_config.json')
if not os.path.isfile(_cfg):
    _cfg = next(iter(glob.glob('/kaggle/input/**/phase3_sasv_config.json', recursive=True)), None)
assert _cfg, 'phase3_sasv_config.json not found - run cell 7 or attach ssl-aasist-phase3-sasv'
CFG = json.load(open(_cfg))
print('Phase 3 config:', _cfg)

if 'linear_fusion' not in dir():
    cal_asv, cal_cm = LLRCalibrator(), LLRCalibrator()
    cal_asv.cal, cal_cm.cal = CFG['llr_asv'], CFG['llr_cm']
    linear_fusion = LinearLLRFusion()
    linear_fusion.w1 = CFG['fusion']['linear']['w_asv']
    linear_fusion.w2 = CFG['fusion']['linear']['w_cm']
    linear_fusion.b = CFG['fusion']['linear']['b']
    cascaded_gate = CascadedMinGate(CFG['fusion']['cascaded_min_gate']['tau_asv'],
                                    CFG['fusion']['cascaded_min_gate']['tau_cm'])
    mlp_fusion = MLPFusionHead()
    mlp_fusion.load_state_dict(torch.load(
        os.path.join(os.path.dirname(_cfg), 'phase3_mlp_fusion.pt'),
        map_location='cpu', weights_only=False)['state_dict'])
    mlp_fusion.eval()
    print('fusion weights rebuilt from config')

assert CM_READY and ASV_READY, 'models not loaded - attach ssl-aasist-phase2j (or 2i/2h/2g/2f) and re-run cell 1'
JOINT_TAU = float(CFG.get('joint_threshold', 0.0))

@torch.no_grad()
def sasv_decide(enroll_path, test_path, verbose=True):
    '''Full SASV decision for one (enrollment, claimant) pair.

    Returns dict of raw scores, calibrated LLRs and per-strategy verdicts.
    accept iff sasv_score >= JOINT_TAU (fixed operational threshold, no hindsight).
    '''
    s_asv, s_cm = score_trial(load_wav_16k(enroll_path), load_wav_16k(test_path))
    la = float(cal_asv.transform(np.array([s_asv]))[0])
    lc = float(cal_cm.transform(np.array([s_cm]))[0])
    a, c = np.array([la]), np.array([lc])
    out = {'s_asv_cos': s_asv, 's_cm_logit': s_cm,
           'llr_asv': la, 'llr_cm': lc,
           'S1_linear': float(linear_fusion.predict_score(a, c)[0]),
           'S2_min_gate': float(cascaded_gate.predict_score(a, c)[0]),
           'S3_mlp': float(mlp_fusion.predict_score(a, c)[0])}
    for k in ('S1_linear', 'S2_min_gate', 'S3_mlp'):
        out[k + '_accept'] = bool(out[k] >= JOINT_TAU)
    if verbose:
        print(f'enroll : {enroll_path}\nclaim  : {test_path}')
        print(f'  cosine sim {s_asv:+.3f} -> LLR_asv {la:+.3f} | '
              f'bona-fide logit {s_cm:+.3f} -> LLR_cm {lc:+.3f}')
        for k in ('S1_linear', 'S2_min_gate', 'S3_mlp'):
            print(f'  {k:12s} score {out[k]:+7.3f} >= {JOINT_TAU:.2f} -> '
                  f'{"ACCEPT" if out[k + "_accept"] else "REJECT"}')
    return out

def sasv_batch(rows):
    '''rows: iterable of (enroll_path, test_path) -> DataFrame-style list of dicts.'''
    return [dict(enroll=e, test=t, **sasv_decide(e, t, verbose=False)) for e, t in rows]

print(f'Inference ready. Joint accept threshold = {JOINT_TAU:.2f}')
print('Usage: sasv_decide("/kaggle/input/<enroll.wav>", "/kaggle/input/<claim.wav>")')

Phase 3 config: /kaggle/working/phase3_out/phase3_sasv_config.json
Inference ready. Joint accept threshold = 0.00
Usage: sasv_decide("/kaggle/input/<enroll.wav>", "/kaggle/input/<claim.wav>")


In [14]:
# --- inference demo: genuine pair vs spoofed claim ---
# Replace these paths with your own enrollment + claimant audio
# (attach any dataset or drop files under /kaggle/working).
_demo = [
    # ('/kaggle/input/<enroll_a.wav>', '/kaggle/input/<claim_same_speaker.wav>'),   # target
    # ('/kaggle/input/<enroll_a.wav>', '/kaggle/input/<claim_other_speaker.wav>'),  # impostor
    # ('/kaggle/input/<enroll_a.wav>', '/kaggle/input/<claim_deepfake.wav>'),       # spoof
]
if _demo:
    for r in sasv_batch(_demo):
        print(r)
else:
    print('no demo pairs configured - edit _demo with real enrollment/claimant paths')

no demo pairs configured - edit _demo with real enrollment/claimant paths


## Reading the result / next step

- **Deployment strategy** is the one with the lowest `actual a-DCF` at the *fixed* joint
  threshold 0 — not the lowest `min a-DCF`, which is the oracle bound. A strategy whose
  actual ≈ min is well calibrated end-to-end; a large gap means joint re-calibration.
- **Baseline sanity:** both fusion strategies should beat `Standalone ASV` on spoof FA and
  `Standalone CM` on impostor FA; that is the whole point of the joint system.
- **S2 (min-gate)** should show the lowest spoof FA by construction — use it when the cost
  of accepting one deepfake dominates (γ → 1). **S3 (MLP)** can overfit on small val sets;
  trust it only if it generalises to the held-out split.
- **Real trials:** attach `sasv_trials.csv` (columns `split,enroll,test,type`) plus the
  `ssl-aasist-phase2f` dataset to switch from simulated scores to real ECAPA + SSL-AASIST
  scoring — or let cell 4b auto-generate it from attached In-the-Wild / VoxCeleb1 /
  ASVspoof 2019 audio. The generated CSV lands in `/kaggle/working` so it can be
  downloaded and pinned as a dataset input for reproducibility.
- **Phase 3b levers** (roadmap §4): `UNFREEZE_TOP_K = 6–8`, external bona-fide speech
  (VoxCeleb / Common Voice), full mobile codec pool (AMR-NB/WB, Opus, GSM-FR), then ONNX
  export of the locked pipeline for Phase 4.